# Sprint 3: Data Cleaning for Taiwanese Bankruptcy Prediction

## Team 4: Andrea Molina, Demetri Blackwood, Dona Erika Moesha Hettiaratchy, Suhail Ansari

---

## Project Context from Annotated Bibliography

This notebook operationalizes the data cleaning strategy outlined in our team's final project proposal. Key findings that guide our approach:

### Dataset Overview
- **Source**: 2020 Taiwanese Bankruptcy Prediction Dataset (UCI Machine Learning Repository)
- **Scale**: ~6,819 firms with 95+ financial ratio features
- **Class Balance**: Extreme imbalance — bankruptcies are ~3% of observations, non-bankrupt firms are ~97%
- **Target Baseline**: Our goal is to exceed the F₂ = 0.423 benchmark from Wang & Liu (2021)

### Data Quality Considerations from Literature
From our review of Tsai et al. (2014), Wang & Liu (2021), and Dasilas & Rigani (2024):

1. **Missing Data & NaNs**: Financial datasets often contain incomplete records. Dropping rows with missing values preserves data integrity but reduces sample size.
2. **Duplicates**: Real-world financial data may have duplicate firm records due to data entry or multiple reporting cycles.
3. **Misplaced Characters**: Text artifacts can corrupt numeric columns, breaking downstream models.
4. **Feature Correlation**: The 95 financial ratios are highly correlated (~95% according to literature). This demands careful outlier and noise handling.
5. **Feature Scaling**: Financial data mixes small ratios (e.g., 0.01) with massive monetary values (e.g., billions). Scaling is essential for many ML algorithms.
6. **Outliers (Anomalies)**: Extreme values in bankruptcy-adjacent firms can mislead classifiers. Gaussian Anomaly Detection identifies these "microscopic tail" observations.
7. **Heteroscedasticity**: Some observations are noisier or less reliable than others. Our cleaning process should flag this for later weighted training.

---

## Notebook Structure

This notebook follows a **two-phase approach**:

### Phase 1: Core Cleaning (Steps 1–6 of Sprint Plan)
- Load and inspect data
- Handle missing values (NaNs)
- Remove duplicates
- Fix misplaced characters
- Report before/after statistics
- Save cleaned CSV

### Phase 2: Advanced Techniques
- Detect and handle outliers (Gaussian Anomaly Detection)
- Evaluate heteroscedasticity indicators
- Apply feature scaling (0–1 normalization) for financial data
- Generate final production-ready dataset

---

# PHASE 1: CORE DATA CLEANING (Steps 1–6)

## Step 1: Set Up Workspace and Libraries

In [ ]:
## ===============================================================
## Import Key Libraries
## ===============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

## Define data paths (relative paths so all teammates can run this)
raw_data_path = "./data/raw_data/data.csv"
cleaned_data_path = "./data/cleaned_data/cleaned_data.csv"
scaled_data_path = "./data/cleaned_data/scaled_data.csv"

## Create output directory if it does not exist
os.makedirs("./data/cleaned_data", exist_ok=True)

print("Libraries imported successfully.")
print(f"Raw data path: {raw_data_path}")
print(f"Cleaned data will be saved to: {cleaned_data_path}")
print(f"Scaled data will be saved to: {scaled_data_path}")

## Step 2: Load and Inspect Data

In [ ]:
## ===============================================================
## Load the Raw Data
## ===============================================================
df_raw = pd.read_csv(raw_data_path)

## Store original shape for comparison
original_shape = df_raw.shape

print("\n" + "="*70)
print("RAW DATA INSPECTION")
print("="*70)
print(f"\nDataset Shape: {original_shape[0]} rows × {original_shape[1]} columns")
print(f"\nFirst 5 Rows:")
df_raw.head()

In [ ]:
## ===============================================================
## Check Data Types and Missing Values
## ===============================================================
print("\nData Types:")
print(df_raw.dtypes.value_counts())

print("\n" + "="*70)
print("Missing Data Summary")
print("="*70)
missing_summary = pd.DataFrame({
    'Column': df_raw.columns,
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Percent': (df_raw.isnull().sum() / len(df_raw)) * 100
})
print(f"\nTotal rows with at least one NaN: {df_raw.isnull().any(axis=1).sum()}")
print(f"\nColumns with missing values:")
print(missing_summary[missing_summary['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False))

In [ ]:
## ===============================================================
## Check Target Variable Balance
## ===============================================================
print("\n" + "="*70)
print("Target Variable Distribution (Bankrupt?)")
print("="*70)
target_counts = df_raw['Bankrupt?'].value_counts()
target_pct = df_raw['Bankrupt?'].value_counts(normalize=True) * 100

balance_df = pd.DataFrame({
    'Class': ['Non-Bankrupt (0)', 'Bankrupt (1)'],
    'Count': [target_counts[0], target_counts[1]],
    'Percentage': [target_pct[0], target_pct[1]]
})
print("\n", balance_df.to_string(index=False))
print(f"\nClass Imbalance Ratio: {target_counts[0] / target_counts[1]:.2f}:1 (non-bankrupt:bankrupt)")

In [ ]:
## ===============================================================
## Visualize Class Imbalance
## ===============================================================
## WHY: A model that always predicts non-bankrupt gets ~97% accuracy
## but detects zero bankruptcies. Standard accuracy is meaningless
## here. Wang & Liu (2021) use F2 (beta=2) which weights recall
## twice as heavily as precision -- missing a bankrupt firm costs
## more than a false alarm. Their best result: F2 = 0.423.

fig, ax = plt.subplots(figsize=(6, 4))
classes = ['Non-Bankrupt (0)', 'Bankrupt (1)']
counts  = [target_counts[0], target_counts[1]]
colors  = ['#1f3e6e', '#d5c164']
bars    = ax.bar(classes, counts, color=colors, width=0.5)

## Label each bar with count and percentage
for bar, count, pct in zip(bars, counts, [target_pct[0], target_pct[1]]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 40,
            f'{count:,}\n({pct:.2f}%)',
            ha='center', va='bottom', fontsize=10)

ax.set_title('Class Imbalance: Bankrupt vs Non-Bankrupt Firms\n(Wang & Liu 2021 baseline: F2 = 0.423)')
ax.set_ylabel('Number of Firms')
ax.set_ylim(0, max(counts) * 1.2)
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()

print(f'Imbalance ratio: {target_counts[0] / target_counts[1]:.1f}:1')
print(f'Naive accuracy ceiling: {target_pct[0]:.2f}% (always predict non-bankrupt)')
print('This is why we benchmark on F2, not accuracy.')

## Step 3: Execute Core Data Cleaning

In [ ]:
## ===============================================================
## Handling Missing Data: Drop Rows with NaNs
## ===============================================================
print("\n" + "="*70)
print("PHASE 1: HANDLING MISSING DATA")
print("="*70)

## Create a working copy
df_clean = df_raw.copy()

rows_before_na = len(df_clean)
df_clean = df_clean.dropna()
rows_after_na = len(df_clean)
rows_removed_na = rows_before_na - rows_after_na

print(f"\nRows before dropna(): {rows_before_na}")
print(f"Rows after dropna(): {rows_after_na}")
print(f"Rows removed due to NaNs: {rows_removed_na}")
print(f"Data retention: {(rows_after_na / rows_before_na) * 100:.2f}%")
print(f"\nJustification: Missing financial ratios cannot be accurately imputed.")
print(f"Dropping incomplete records preserves data integrity while retaining ")
print(f"{(rows_after_na / rows_before_na) * 100:.1f}% of the dataset.")

In [ ]:
## ===============================================================
## Handling Duplicates
## ===============================================================
print("\n" + "="*70)
print("PHASE 1: HANDLING DUPLICATES")
print("="*70)

rows_before_dup = len(df_clean)
df_clean = df_clean.drop_duplicates()
rows_after_dup = len(df_clean)
rows_removed_dup = rows_before_dup - rows_after_dup

print(f"\nRows before drop_duplicates(): {rows_before_dup}")
print(f"Rows after drop_duplicates(): {rows_after_dup}")
print(f"Duplicate rows removed: {rows_removed_dup}")
if rows_removed_dup > 0:
    print(f"\nJustification: Duplicate firm records inflate model training and can lead")
    print(f"to inflated performance metrics. Removing {rows_removed_dup} duplicates prevents this bias.")
else:
    print(f"\nNo duplicates detected in the dataset.")

In [ ]:
## ===============================================================
## Correcting Data Types and Fixing Misplaced Characters
## ===============================================================
print("\n" + "="*70)
print("PHASE 1: CORRECTING DATA TYPES AND MISPLACED CHARACTERS")
print("="*70)

## Identify non-numeric columns (excluding target variable)
print("\nChecking for non-numeric columns...")
non_numeric_cols = []

for col in df_clean.columns:
    if col != 'Bankrupt?':
        try:
            pd.to_numeric(df_clean[col])
        except (ValueError, TypeError):
            non_numeric_cols.append(col)

if non_numeric_cols:
    print(f"\nFound {len(non_numeric_cols)} non-numeric columns:")
    for col in non_numeric_cols[:10]:  # Show first 10
        print(f"  - {col}")
    print("\nAttempting to convert these columns to numeric...")
    for col in non_numeric_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    print("Conversion complete.")
else:
    print("\nAll columns are already numeric. No character correction needed.")

## Verify data types
print("\nFinal data types summary:")
print(df_clean.dtypes.value_counts())

In [ ]:
## ===============================================================
## Final NaN Check After Type Conversion
## ===============================================================
print("\nChecking for NaNs introduced by failed conversions...")
new_nans = df_clean.isnull().sum().sum()
if new_nans > 0:
    print(f"\nFound {new_nans} NaNs introduced during type conversion.")
    print("Dropping these rows...")
    df_clean = df_clean.dropna()
    print(f"Rows remaining: {len(df_clean)}")
else:
    print(f"No additional NaNs. Data is clean.")

## Step 4: Evaluate Cleaned Data

In [ ]:
## ===============================================================
## Before and After Comparison
## ===============================================================
print("\n" + "="*70)
print("CORE CLEANING: BEFORE AND AFTER")
print("="*70)

comparison_df = pd.DataFrame({
    'Metric': ['Rows', 'Columns', 'Memory (MB)'],
    'Before': [
        original_shape[0],
        original_shape[1],
        df_raw.memory_usage(deep=True).sum() / 1e6
    ],
    'After': [
        df_clean.shape[0],
        df_clean.shape[1],
        df_clean.memory_usage(deep=True).sum() / 1e6
    ]
})
comparison_df['Difference'] = comparison_df['Before'] - comparison_df['After']
comparison_df['% Retained'] = (comparison_df['After'] / comparison_df['Before'] * 100).round(2)

print("\n", comparison_df.to_string(index=False))

print(f"\nTotal rows removed: {original_shape[0] - df_clean.shape[0]}")
print(f"Total rows retained: {df_clean.shape[0]}")
print(f"Data retention rate: {(df_clean.shape[0] / original_shape[0]) * 100:.2f}%")

In [ ]:
## ===============================================================
## Post-Cleaning Target Distribution
## ===============================================================
print("\nPost-Cleaning Target Variable Distribution:")
post_clean_counts = df_clean['Bankrupt?'].value_counts()
post_clean_pct = df_clean['Bankrupt?'].value_counts(normalize=True) * 100

post_balance_df = pd.DataFrame({
    'Class': ['Non-Bankrupt (0)', 'Bankrupt (1)'],
    'Count': [post_clean_counts[0], post_clean_counts[1]],
    'Percentage': [post_clean_pct[0], post_clean_pct[1]]
})
print("\n", post_balance_df.to_string(index=False))
print(f"\nClass imbalance ratio: {post_clean_counts[0] / post_clean_counts[1]:.2f}:1")

## Step 5: Save Core Cleaned Data

In [ ]:
## ===============================================================
## Save Cleaned CSV
## ===============================================================
print("\nSaving cleaned data to CSV...")
df_clean.to_csv(cleaned_data_path, index=False)
print(f"\nCleaned data saved to: {cleaned_data_path}")
print(f"File size: {os.path.getsize(cleaned_data_path) / 1e6:.2f} MB")

In [ ]:
## ===============================================================
## Note: Loading Cleaned Data in Sprint 4
## ===============================================================
## Sprint Additional Step 1: when loading the CSV into a raw NumPy
## array you MUST skip the header row. The first row contains column
## name strings, not numbers -- numpy.loadtxt will crash with
## "could not convert string to float" if you omit skiprows=1.

## Option A -- NumPy (skip text header):
## data = np.loadtxt(cleaned_data_path, delimiter=',', skiprows=1)

## Option B -- Pandas then convert (recommended -- keeps column names):
## df = pd.read_csv(cleaned_data_path)
## data = df.to_numpy()

print('Sprint 4 loading options:')
print('  Option A (NumPy):  np.loadtxt(path, delimiter=",", skiprows=1)')
print('  Option B (Pandas): pd.read_csv(path).to_numpy()   <-- recommended')
print(f'Cleaned data ready: {df_clean.shape[0]} rows x {df_clean.shape[1]} columns')

## Step 6: Core Cleaning Summary Report

### Summary of Core Cleaning Actions

| Action | Rows Removed | Justification |
|--------|-------------|---------------|
| **Drop NaNs** | {rows_removed_na} | Missing financial ratios cannot be accurately imputed. |
| **Drop Duplicates** | {rows_removed_dup} | Duplicate firm records inflate model training. |
| **Fix Data Types** | {new_nans} | Character errors corrected; failed conversions dropped. |
| **Total Removed** | **{original_shape[0] - df_clean.shape[0]}** | **Data retention: {(df_clean.shape[0] / original_shape[0]) * 100:.2f}%** |

### Key Findings

- **Original dataset**: {original_shape[0]:,} firms × {original_shape[1]} features
- **Cleaned dataset**: {df_clean.shape[0]:,} firms × {df_clean.shape[1]} features
- **Class balance**: {post_clean_counts[0]:,} non-bankrupt, {post_clean_counts[1]:,} bankrupt (imbalance ratio: {post_clean_counts[0] / post_clean_counts[1]:.2f}:1)
- **Data quality**: All columns are now numeric and free of missing values.

---

In [ ]:
## ===============================================================
## Correlation Analysis: Quantify Collinear Features
## ===============================================================
## Gap identified in Wang & Liu (2021): no feature selection was
## applied to the 95 financial ratios despite high collinearity.
## This cell QUANTIFIES the redundancy. Feature removal (PCA or
## correlation-based pruning) will happen in Sprint 4. The numbers
## here directly support the feature selection argument in the paper.
##
## Method: Pearson correlation between every feature pair.
## Threshold |r| > 0.9 means two features carry nearly identical
## information. We loop over the upper triangle (i < j) to count
## each pair exactly once (not twice).

## Redefine feature columns here -- Phase 2 metadata cols not added yet
corr_cols = [col for col in df_clean.columns
             if col not in ['Bankrupt?', 'is_outlier', 'noise_proxy', 'is_high_noise']]

## Compute Pearson correlation matrix, convert to numpy for loop
corr_matrix = df_clean[corr_cols].corr().to_numpy()
N = len(corr_cols)

high_corr_count = 0
high_corr_pairs = []

for i in range(N):
    for j in range(i + 1, N):          ## upper triangle avoids double-counting
        r = np.absolute(corr_matrix[i, j])
        if r > 0.9:
            high_corr_count += 1
            high_corr_pairs.append((corr_cols[i], corr_cols[j], corr_matrix[i, j]))

total_pairs = N * (N - 1) // 2

print('=' * 70)
print('CORRELATION ANALYSIS: COLLINEAR FEATURE PAIRS')
print('=' * 70)
print(f'\n  Total features:                     {N}')
print(f'  Total feature pairs:                {total_pairs}')
print(f'  Highly collinear pairs (|r| > 0.9): {high_corr_count}')
print(f'  Percentage collinear:               {high_corr_count / total_pairs * 100:.1f}%')

## Sort by absolute correlation, show top 5 most redundant pairs
high_corr_pairs_sorted = sorted(high_corr_pairs, key=lambda x: np.absolute(x[2]), reverse=True)
print(f'\nTop 5 most correlated pairs:')
for feat1, feat2, r in high_corr_pairs_sorted[:5]:
    print(f'  r={r:.4f} | {feat1[:38]:<38} <-> {feat2[:38]}')

print(f'\nSprint 4 implication:')
print(f'  {high_corr_count} redundant pairs found. PCA or correlation-based pruning')
print(f'  should reduce collinear noise before ensemble training,')
print(f'  directly addressing the gap Wang & Liu (2021) left open.')

# PHASE 2: ADVANCED TECHNIQUES

## Context

Our literature review (Dasilas & Rigani, 2024; Wang & Liu, 2021) identifies three advanced data quality challenges:

1. **Outliers/Anomalies**: Extreme values that fall in the "microscopic tail" of the distribution can distort model training.
2. **Heteroscedasticity**: Some observations are noisier or have lower reliability than others—flagging these allows weighted training.
3. **Feature Scaling**: Financial data mixes small ratios with massive monetary values. Scaling to [0, 1] prevents larger numbers from dominating.

These techniques complement core cleaning and enhance model robustness.

## Advanced Technique 1: Detect and Handle Outliers (Gaussian Anomaly Detection)

In [ ]:
## ===============================================================
## Gaussian Anomaly Detection: Identify Statistical Outliers
## ===============================================================
print("\n" + "="*70)
print("ADVANCED TECHNIQUE 1: GAUSSIAN ANOMALY DETECTION")
print("="*70)

print("\nMethod: Calculate Z-scores for each feature.")
print("Rationale: Under a Gaussian (normal) distribution, values with |Z| > 3")
print("are considered extreme outliers (~0.27% probability).")

## Select all feature columns (everything except the target)
feature_cols = [col for col in df_clean.columns if col != "Bankrupt?"]

## Compute Z-scores manually using numpy (no scipy needed)
## Z = (x - mean) / std for each column
feature_data = df_clean[feature_cols].to_numpy()
col_means = np.zeros(feature_data.shape[1])
col_stds = np.zeros(feature_data.shape[1])

for k in range(feature_data.shape[1]):
    col_means[k] = np.mean(feature_data[:, k])
    col_stds[k] = np.std(feature_data[:, k])

## Avoid division by zero for constant columns
col_stds[col_stds == 0] = 1

z_scores = np.absolute((feature_data - col_means) / col_stds)

## Flag any row where at least one feature has |Z| > 3
outlier_mask = (z_scores > 3).any(axis=1)
num_outliers = np.sum(outlier_mask)

print(f"\nRows with at least one feature |Z| > 3: {num_outliers}")
print(f"Percentage of data: {(num_outliers / len(df_clean)) * 100:.2f}%")

outlier_rows = df_clean[outlier_mask]
print(f"\n  Non-bankrupt outliers: {(outlier_rows['Bankrupt?'] == 0).sum()}")
print(f"  Bankrupt outliers:     {(outlier_rows['Bankrupt?'] == 1).sum()}")

In [ ]:
## ===============================================================
## Option: Remove or Flag Outliers
## ===============================================================
print("\n" + "-"*70)
print("Outlier Handling Strategy")
print("-"*70)

## For this project, we FLAG outliers rather than removing them
## This allows downstream models to apply weighted training if desired
df_clean['is_outlier'] = outlier_mask.astype(int)

print("\nDecision: FLAGGED (not removed)")
print("Rationale: Outliers may represent rare but genuine bankruptcy patterns.")
print("By flagging them, downstream models can:")
print("  1. Apply weighted training (lower weight to outliers)")
print("  2. Investigate edge cases for domain insight")
print("  3. Perform separate analysis on outlier vs. normal observations")

print(f"\nOutlier flag added to column: is_outlier")
print(f"Value 1 = outlier, Value 0 = normal observation")

## Advanced Technique 2: Evaluate Heteroscedasticity Indicators

In [ ]:
## ===============================================================
## Heteroscedasticity Analysis
## ===============================================================
print("\n" + "="*70)
print("ADVANCED TECHNIQUE 2: HETEROSCEDASTICITY EVALUATION")
print("="*70)

print("\nDefinition: Heteroscedasticity = varying noise/variance across observations.")
print("\nContext: Some firm financial records are noisier or less trustworthy than")
print("others (e.g., firms with incomplete disclosures, merged/acquired firms).")
print("\nApproach: Compute residual noise proxy for each observation.")

## Compute coefficient of variation (CV) for each row's features
## High CV = high relative variability = potentially noisy observation
df_hetero = df_clean[feature_cols].copy()
row_means = df_hetero.mean(axis=1)
row_stds = df_hetero.std(axis=1)
row_cv = row_stds / (row_means + 1e-8)  # Avoid division by zero

df_clean['noise_proxy'] = row_cv

print(f"\nNoise Proxy (Coefficient of Variation) Statistics:")
print(f"  Mean: {row_cv.mean():.4f}")
print(f"  Std:  {row_cv.std():.4f}")
print(f"  Min:  {row_cv.min():.4f}")
print(f"  Max:  {row_cv.max():.4f}")
print(f"  Q1:   {row_cv.quantile(0.25):.4f}")
print(f"  Q3:   {row_cv.quantile(0.75):.4f}")

## Identify high-noise observations (top quartile)
high_noise_threshold = row_cv.quantile(0.75)
high_noise_mask = row_cv > high_noise_threshold
print(f"\nHigh-Noise Observations (top quartile, CV > {high_noise_threshold:.4f}):")
print(f"  Count: {high_noise_mask.sum()}")
print(f"  Percentage: {(high_noise_mask.sum() / len(df_clean)) * 100:.2f}%")

In [ ]:
## ===============================================================
## Flag High-Noise Observations
## ===============================================================
df_clean['is_high_noise'] = high_noise_mask.astype(int)

print("\nDecision: FLAGGED high-noise observations")
print("Rationale: These observations exhibit high relative variability in their")
print("financial ratios, suggesting potential measurement noise or reporting issues.")
print("\nDownstream Use:")
print("  - Apply lower sample weights during model training")
print("  - Use separate validation sets for stable vs. noisy obs.")
print("  - Investigate data quality flags at source")

## Advanced Technique 3: Feature Scaling (0–1 Normalization)

In [ ]:
## ===============================================================
## Feature Scaling: Min-Max Normalization [0, 1]
## ===============================================================
print("\n" + "="*70)
print("ADVANCED TECHNIQUE 3: FEATURE SCALING (MIN-MAX NORMALIZATION)")
print("="*70)

print("\nRationale: Financial data mixes small ratios (e.g., 0.01 efficiency)")
print("with massive monetary values (e.g., 9.15 billion in assets).")
print("\nWithout scaling:")
print("  - Gradient-based models learn biased gradients (large values dominate)")
print("  - Distance-based models (KNN, SVM) give unequal importance")
print("  - Regularization penalties are applied inconsistently")
print("\nSolution: Min-Max normalization to [0, 1] range.")
print("Formula: X_scaled = (X - X_min) / (X_max - X_min)")

## Create scaled copy
df_scaled = df_clean.copy()

## Scale only feature columns (not target, outlier flag, or noise proxy)
cols_to_scale = feature_cols
scaler_params = {}

for col in cols_to_scale:
    col_min = df_clean[col].min()
    col_max = df_clean[col].max()
    col_range = col_max - col_min
    
    if col_range == 0:  # Handle constant columns
        df_scaled[col] = 0.5  # Constant features -> 0.5
    else:
        df_scaled[col] = (df_clean[col] - col_min) / col_range
    
    scaler_params[col] = {'min': col_min, 'max': col_max}

print(f"\nScaling complete for {len(cols_to_scale)} features.")
print(f"\nSample scaling statistics (first 5 features):")
for i, col in enumerate(cols_to_scale[:5]):
    print(f"  {col}:")
    print(f"    Original range: [{scaler_params[col]['min']:.6f}, {scaler_params[col]['max']:.6f}]")
    print(f"    Scaled range: [{df_scaled[col].min():.4f}, {df_scaled[col].max():.4f}]")

In [ ]:
## ===============================================================
## Verify Scaling
## ===============================================================
print("\nVerifying scaled data:")
print(f"\nScaled data shape: {df_scaled.shape}")
print(f"Columns: Target + {len(cols_to_scale)} scaled features + metadata (is_outlier, noise_proxy, is_high_noise)")

print("\nScaled feature ranges (all should be [0, 1]):")
scaled_feature_stats = pd.DataFrame({
    'Feature': cols_to_scale[:10],  # Show first 10
    'Min': [df_scaled[col].min() for col in cols_to_scale[:10]],
    'Max': [df_scaled[col].max() for col in cols_to_scale[:10]],
    'Mean': [df_scaled[col].mean() for col in cols_to_scale[:10]]
})
print("\n", scaled_feature_stats.to_string(index=False))

## Final: Save Advanced-Cleaned (Scaled) Data

In [ ]:
## ===============================================================
## Save Scaled Data
## ===============================================================
print("\n" + "="*70)
print("SAVING FINAL CLEANED & SCALED DATA")
print("="*70)

df_scaled.to_csv(scaled_data_path, index=False)
print(f"\nScaled data saved to: {scaled_data_path}")
print(f"File size: {os.path.getsize(scaled_data_path) / 1e6:.2f} MB")

print(f"\nFinal dataset summary:")
print(f"  Rows: {df_scaled.shape[0]}")
print(f"  Columns: {df_scaled.shape[1]}")
print(f"  Features (scaled): {len(cols_to_scale)}")
print(f"  Metadata: is_outlier, noise_proxy, is_high_noise")
print(f"  Target: Bankrupt? (0=No, 1=Yes)")

---

# Final Cleaning Summary and Recommendations

## What Was Done

### Core Phase (Steps 1–6)
✓ Dropped {rows_removed_na} rows with missing values  
✓ Removed {rows_removed_dup} duplicate firm records  
✓ Corrected data types; converted text fields to numeric  
✓ Final dataset: {df_clean.shape[0]:,} firms × {df_clean.shape[1]} columns  
✓ Data retention: {(df_clean.shape[0] / original_shape[0]) * 100:.2f}%

### Advanced Phase
✓ Applied Gaussian Anomaly Detection (Z-score > 3)  
✓ Flagged {num_outliers} outlier observations for weighted training  
✓ Computed heteroscedasticity proxy (noise_proxy column)  
✓ Identified {high_noise_mask.sum()} high-noise observations  
✓ Applied Min-Max scaling [0, 1] to all {len(cols_to_scale)} features  

## Deliverables

Three datasets saved to `./data/cleaned_data/`:

1. **cleaned_data.csv**  
   - Core-cleaned data with flagged outliers and noise proxy
   - Use for EDA and interpretability analysis

2. **scaled_data.csv**  
   - Production-ready for model training (gradient boosting, neural networks)
   - All features normalized [0, 1]
   - Metadata columns: is_outlier, noise_proxy, is_high_noise

3. **Raw data backup**  
   - Original data retained in `./data/raw_data/data.csv`
   - Ensures reproducibility and audit trail

## Next Steps (Sprint 4: Modeling)

1. **Use scaled_data.csv** for training heterogeneous ensembles and tuned gradient boosting
2. **Leverage is_outlier flag** for weighted training (Wang & Liu approach)
3. **Apply noise_proxy weighting** in loss functions to penalize model errors on high-noise observations
4. **Benchmark against F₂ = 0.423** baseline from prior literature
5. **Monitor target class imbalance** (96.77% non-bankrupt, 3.23% bankrupt); consider undersampling (Tomek Links, ENN) during model training

---

## Data Cleaning Completion Status

✅ **All core and advanced cleaning steps completed**

The Taiwanese Bankruptcy Prediction dataset is now ready for model training in Sprint 4. The combination of core cleaning (NaN/duplicate handling) and advanced techniques (outlier detection, scaling, noise flagging) provides a robust, production-grade dataset that balances:

- **Data integrity** (removed corrupted/incomplete records)
- **Interpretability** (outlier/noise flags allow investigation)
- **Model robustness** (scaled features, imbalance awareness)
- **Audit trail** (original data preserved, all changes logged)

**Data is ready for transfer to modeling pipeline.**